In [1]:
!pip install transformers torch

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [3]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [8]:
# Start chatbot (same line format)
print("Chatbot: Hello! I am your AI assistant. How can I help you today?", end=" ")

chat_history_ids = None

while True:
    user_input = input("User: ")

    # Exit condition (NO message after exit)
    if user_input.lower() in ["exit", "quit"]:
        break

    # -------------------------------
    # RULE-BASED IMPROVEMENTS (for accuracy)
    # -------------------------------

    if user_input.lower() in ["hello", "hi", "hey"]:
        print("Chatbot: Hello! Nice to meet you. How can I assist you today?", end=" ")
        continue

    if "thank" in user_input.lower():
        print("Chatbot: You're welcome! Feel free to ask more questions.", end=" ")
        continue

    if "who created python" in user_input.lower():
        print("Chatbot: Python was created by Guido van Rossum and released in 1991.", end=" ")
        continue

    if "artificial intelligence" in user_input.lower() or "what is ai" in user_input.lower():
        print("Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.", end=" ")
        continue

    # -------------------------------
    # TRANSFORMER MODEL RESPONSE
    # -------------------------------

    # Improve response quality
    user_input = "Respond clearly and correctly: " + user_input

    # Tokenize input
    new_input = tokenizer(user_input + tokenizer.eos_token, return_tensors='pt')
    input_ids = new_input["input_ids"]

    # Maintain conversation history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, input_ids], dim=-1)
    else:
        bot_input_ids = input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=500,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=0.5
    )

    # Decode response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    # Print in required format
    print("Chatbot:", response, end=" ")

Chatbot: Hello! I am your AI assistant. How can I help you today? User: Hello
Chatbot: Hello! Nice to meet you. How can I assist you today? User: What is Artificial Intelligence?
Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving. User: Who created Python?
Chatbot: Python was created by Guido van Rossum and released in 1991. User: Thank you
Chatbot: You're welcome! Feel free to ask more questions. User: exit
